## 3. Анализ ревизии склада

**Источник данных:** сличительная ведомость из системы iiko в формате PDF, 28.03.2026  
**Проблемы при загрузке:** таблица в PDF с объединёнными ячейками в заголовке,  
числа с пробелами и запятыми вместо точек  
**Итоговый датасет:** 231 позиция, ООО «Искусство», кухня

**Вопросы исследования:**
- Где наибольшие расхождения между фактом и расчётом?
- Какие позиции уходят в отрицательный расчётный остаток?
- Где деньги теряются — в недостачах или в неправильных ТТК?

In [ ]:
# Устанавливаем библиотеку для чтения PDF-таблиц
# Запусти эту ячейку один раз
import subprocess
subprocess.run(['pip', 'install', 'pdfplumber'], capture_output=True)
print('Готово')

In [ ]:
import pdfplumber
import pandas as pd
import matplotlib.pyplot as plt

# Загружаем PDF
rows = []
with pdfplumber.open('../data/Сличительная_ведомость.pdf') as pdf:
    for page in pdf.pages:
        table = page.extract_table()
        if table:
            rows.extend(table)

# Пропускаем первые 3 строки-заголовка
data_rows = rows[3:]

records = []
for row in data_rows:
    if not row[0] or row[0] == 'Итого':
        continue
    try:
        int(row[0])
    except:
        continue
    
    records.append({
        'num':          row[0],
        'name':         row[1],
        'code':         row[2],
        'unit':         row[3],
        'price':        row[4],
        'fact_qty':     row[5],
        'fact_sum':     row[6],
        'calc_qty':     row[7],
        'calc_sum':     row[8],
        'surplus_qty':  row[9],
        'surplus_sum':  row[10],
        'shortage_qty': row[11],
        'shortage_sum': row[12],
    })

df_inv = pd.DataFrame(records)

num_cols = ['price','fact_qty','fact_sum','calc_qty','calc_sum',
            'surplus_qty','surplus_sum','shortage_qty','shortage_sum']

for col in num_cols:
    df_inv[col] = (df_inv[col]
                   .astype(str)
                   .str.replace(' ', '', regex=False)
                   .str.replace(',', '.', regex=False)
                   .replace('', '0')
                   .replace('None', '0'))
    df_inv[col] = pd.to_numeric(df_inv[col], errors='coerce').fillna(0)

# Вспомогательная функция — используется в графиках
def clean_name(name):
    return (name.replace(', кг.', '')
                .replace(', шт.', '')
                .replace(', л.', '')
                .replace(' св.', '')
                .replace(' с/м', ' с/м'))

# Производные переменные
has_shortage = df_inv[df_inv['shortage_sum'] > 0].copy()
has_surplus  = df_inv[df_inv['surplus_sum']  > 0].copy()
df_neg       = df_inv[df_inv['calc_qty'] < 0].copy()
df_neg['short_name'] = df_neg['name'].apply(clean_name)
df_neg = df_neg.sort_values('calc_qty', ascending=True)

print(f'Позиций загружено: {len(df_inv)}')

In [ ]:
# Общая статистика
total_shortage = df_inv['shortage_sum'].sum()
total_surplus  = df_inv['surplus_sum'].sum()
net            = total_surplus - total_shortage

print('='*45)
print('ИТОГИ РЕВИЗИИ')
print('='*45)
print(f'Суммарные излишки:   {total_surplus:>10,.0f} руб')
print(f'Суммарная недостача: {total_shortage:>10,.0f} руб')
print(f'Нетто-позиция:       {net:>+10,.0f} руб')
print()

# Позиции с недостачей
has_shortage = df_inv[df_inv['shortage_sum'] > 0].copy()
has_surplus  = df_inv[df_inv['surplus_sum']  > 0].copy()

print(f'Позиций с недостачей: {len(has_shortage)} из {len(df_inv)}')
print(f'Позиций с излишками:  {len(has_surplus)} из {len(df_inv)}')
print()

# Процент недостачи от фактического остатка
total_fact = df_inv['fact_sum'].sum()
print(f'Фактические остатки: {total_fact:>10,.0f} руб')
print(f'% недостачи от факта: {total_shortage/total_fact*100:.1f}%')
print(f'% излишков от факта:  {total_surplus/total_fact*100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ── Топ-15 недостач (справа должны быть большие) ─────────
top_sh = (df_inv[df_inv['shortage_sum'] > 0]
          .sort_values('shortage_sum', ascending=True)
          .tail(15)
          .copy())
top_sh['short_name'] = top_sh['name'].apply(clean_name)

# ── Топ-15 излишков ───────────────────────────────────────
top_su = (df_inv[df_inv['surplus_sum'] > 0]
          .sort_values('surplus_sum', ascending=True)
          .tail(15)
          .copy())
top_su['short_name'] = top_su['name'].apply(clean_name)

# Недостачи — слева
bars1 = axes[0].barh(top_sh['short_name'], top_sh['shortage_sum'],
                     color='salmon', edgecolor='white')
for bar, val in zip(bars1, top_sh['shortage_sum']):
    axes[0].text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2,
                 f'{val:,.0f} ₽', va='center', fontsize=9)
axes[0].set_title('Топ-15 недостач по сумме', fontsize=13, pad=12)
axes[0].set_xlabel('Сумма недостачи, руб')
axes[0].grid(axis='x', alpha=0.3)
axes[0].spines[['top','right']].set_visible(False)
axes[0].set_xlim(0, top_sh['shortage_sum'].max() * 1.25)

# Излишки — справа
bars2 = axes[1].barh(top_su['short_name'], top_su['surplus_sum'],
                     color='steelblue', edgecolor='white')
for bar, val in zip(bars2, top_su['surplus_sum']):
    axes[1].text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2,
                 f'{val:,.0f} ₽', va='center', fontsize=9)
axes[1].set_title('Топ-15 излишков по сумме', fontsize=13, pad=12)
axes[1].set_xlabel('Сумма излишков, руб')
axes[1].grid(axis='x', alpha=0.3)
axes[1].spines[['top','right']].set_visible(False)
axes[1].set_xlim(0, top_su['surplus_sum'].max() * 1.25)

plt.suptitle('Анализ ревизии — Ресторан Искусство, 28.03.2026',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Берём только топ-15 по абсолютному значению отклонения
df_neg_top = df_neg.nsmallest(15, 'calc_qty').copy()

fig, ax = plt.subplots(figsize=(13, 7))

bars = ax.barh(df_neg_top['short_name'], df_neg_top['calc_qty'],
               color='firebrick', edgecolor='white')

for bar, val in zip(bars, df_neg_top['calc_qty']):
    ax.text(bar.get_width() - 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', ha='right',
            fontsize=10, color='white', fontweight='bold')

ax.axvline(0, color='black', linewidth=1)
ax.set_title('Топ-15 позиций с отрицательным расчётным остатком\n'
             '(система продала больше чем было оприходовано)',
             fontsize=13, pad=12)
ax.set_xlabel('Расчётный остаток (кол-во единиц)')
ax.tick_params(axis='y', labelsize=11)
ax.grid(axis='x', alpha=0.3)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

print(f'\nВсего позиций в минусе: {len(df_neg)} из {len(df_inv)}')

In [ ]:
fig = plt.figure(figsize=(18, 12))
fig.suptitle('Анализ ревизии — Ресторан Искусство, 28.03.2026',
             fontsize=15, fontweight='bold', y=1.01)

# ── График 1: общая картина (pie) ─────────────────────────
ax1 = fig.add_subplot(2, 3, 1)
values = [total_shortage, total_surplus]
labels = [f'Недостача\n{total_shortage:,.0f} ₽', 
          f'Излишки\n{total_surplus:,.0f} ₽']
colors = ['salmon', 'steelblue']
wedges, texts = ax1.pie(values, labels=labels, colors=colors,
                         startangle=90,
                         wedgeprops={'edgecolor':'white', 'linewidth':2})
for text in texts:
    text.set_fontsize(10)
ax1.set_title('Соотношение\nизлишков и недостач', fontsize=11, pad=10)

# ── График 2: топ-10 недостач ─────────────────────────────
ax2 = fig.add_subplot(2, 3, (2, 3))
top10_sh = (df_inv[df_inv['shortage_sum'] > 0]
            .sort_values('shortage_sum', ascending=True)
            .tail(10).copy())
top10_sh['short_name'] = top10_sh['name'].apply(clean_name)
bars = ax2.barh(top10_sh['short_name'], top10_sh['shortage_sum'],
                color='salmon', edgecolor='white')
for bar, val in zip(bars, top10_sh['shortage_sum']):
    ax2.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
             f'{val:,.0f} ₽', va='center', fontsize=9)
ax2.set_title('Топ-10 недостач', fontsize=11, pad=10)
ax2.set_xlabel('Сумма, руб')
ax2.grid(axis='x', alpha=0.3)
ax2.spines[['top','right']].set_visible(False)
ax2.set_xlim(0, top10_sh['shortage_sum'].max() * 1.2)

# ── График 3: топ-10 излишков ─────────────────────────────
ax3 = fig.add_subplot(2, 3, (4, 5))
top10_su = (df_inv[df_inv['surplus_sum'] > 0]
            .sort_values('surplus_sum', ascending=True)
            .tail(10).copy())
top10_su['short_name'] = top10_su['name'].apply(clean_name)
bars = ax3.barh(top10_su['short_name'], top10_su['surplus_sum'],
                color='steelblue', edgecolor='white')
for bar, val in zip(bars, top10_su['surplus_sum']):
    ax3.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
             f'{val:,.0f} ₽', va='center', fontsize=9)
ax3.set_title('Топ-10 излишков', fontsize=11, pad=10)
ax3.set_xlabel('Сумма, руб')
ax3.grid(axis='x', alpha=0.3)
ax3.spines[['top','right']].set_visible(False)
ax3.set_xlim(0, top10_su['surplus_sum'].max() * 1.2)

# ── График 4: статистика ──────────────────────────────────
ax4 = fig.add_subplot(2, 3, 6)
ax4.axis('off')
stats = [
    ['Всего позиций',         f'{len(df_inv)}'],
    ['С недостачей',          f'{len(has_shortage)} поз.'],
    ['С излишками',           f'{len(has_surplus)} поз.'],
    ['Сумма недостач',        f'{total_shortage:,.0f} ₽'],
    ['Сумма излишков',        f'{total_surplus:,.0f} ₽'],
    ['Нетто-позиция',         f'+{net:,.0f} ₽'],
    ['% недостачи от факта',  f'{total_shortage/total_fact*100:.1f}%'],
    ['В минусе (ТТК)',         f'{len(df_neg)} поз.'],
]
table = ax4.table(cellText=stats,
                  colLabels=['Показатель', 'Значение'],
                  cellLoc='left', loc='center',
                  bbox=[0, 0, 1, 1])
table.auto_set_font_size(False)
table.set_fontsize(10)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('steelblue')
        cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#f0f4f8')
    cell.set_edgecolor('white')
ax4.set_title('Сводная статистика', fontsize=11, pad=10)

plt.tight_layout()
plt.show()

## Выводы — Анализ ревизии

**Данные:** сличительная ведомость продуктов от 28.03.2026, 231 позиция, ООО «Искусство»

### Общая картина
- Излишки: 98,610 ₽ — Недостача: 44,335 ₽ — Нетто: +54,275 ₽
- 21.7% недостачи от фактических остатков — требует внимания
- Большие излишки при больших недостачах = проблема в ТТК

### Топ недостач
- Водоросли нори (3,850 ₽), Свиная шейка (3,428 ₽), Сливки 33% (3,096 ₽)
- Расходные позиции с высокой оборачиваемостью — нормы расхода занижены

### Топ излишков
- Ягнёнок (11,507 ₽), Говяжья вырезка (8,903 ₽), Лопатка свиная (7,991 ₽)
- Дорогое мясо не списывается системой — ТТК не подключены или блюда убраны из меню

### Аномалии — 35 позиций в минусе
- Яйцо перепелиное: − 100 шт — приход не проведён или не списаны
- Булочки для гамбургера: − 56 шт — не были оприходованны
- Вода питьевая: − 64 кг — убрать из ТТК

### Рекомендации
1. Пересмотреть ТТК блюд, учесть продукты из списка недостач, пересчитать повторно ревизию
2. Сделать пересчёт ТТК блюд, в которых используется продукты из списка излишков
3. Проверить приходные накладные по аномальным позициям
4. Вести строгий учёт списания продуктов, сократить норму продуктов питания персонала;
5. Провести нулевую ревизию и делать её на ежемесячной основе;
6. Проанализировать динамику списаний, оптимизировать частоту и объём заказов и увеличить перекрестное использование продуктов;    